# 14 · Multi-hop Retrieval

**Idea:** some questions need a *chain* of facts — the answer to one lookup tells you what to search for next. Single-shot retrieval fails here. **Multi-hop** loops: retrieve, let an LLM propose the next query to fill the gap, retrieve again, then answer from everything gathered.

We use the `systems_and_teams.md` corpus, where the facts are deliberately spread across sections.

> Runs the loop on Ollama (several small calls).

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [ ]:
from rag_lab import load_markdown, NumpyStore

corpus = load_markdown()
store = NumpyStore(provider='ollama'); store.add(corpus)

# A question no single passage answers:
#   Helpdesk Assistant -> powered by Atlas -> Atlas calls Beacon -> built by Search Platform -> led by Priya Nair
question = 'Who leads the team that built the reranker used by the Helpdesk Assistant?'

First show that **single-hop** retrieval struggles: the top passages rarely contain the whole chain in one place.

In [ ]:
for h in store.search(question, k=3):
    print(f'{h.score:5.2f}  {h.chunk.source} :: {h.chunk.meta.get("heading")}')

Now run multi-hop. Watch the `queries` trace — the LLM reformulates toward the missing fact at each hop.

In [ ]:
from rag_lab import multihop_answer
from study_buddy import CostTracker

tracker = CostTracker()
answer, trace = multihop_answer(question, store, provider='ollama', k=3, max_hops=3, tracker=tracker)

print('hops:')
for i, q in enumerate(trace.queries, 1):
    print(f'  {i}. {q}')
print('\nanswer:', answer)
print('\ncost:', tracker.summary())

**Takeaway:** multi-hop turns retrieval into a loop steered by the model. It unlocks bridge questions that no single chunk answers — at the price of extra retrieval + reasoning calls, so cap `max_hops`.

Next → `15_agentic_rag.ipynb`